# Screening Food Access in Washington, DC

This notebook uses Graphways to build a simple network-based food-access screen. It is not an official USDA food-desert classification: those definitions include income, vehicle access, census geography, and other demographic measures. The goal here is narrower and useful as a first analytical pass:

- build a reusable walking `SpatialGraph` for DC,
- fetch grocery-like OpenStreetMap POIs,
- sample points across the city,
- ask whether each point can reach a grocery store within a walk-time threshold,
- map the likely underserved areas for follow-up analysis.

Install optional notebook mapping dependencies with:

```bash
pip install graphways folium branca
```

In [1]:
import base64
import json
from statistics import median

import branca.colormap
import folium
import graphways as gw
from IPython.display import HTML, display


def show_map(m, height=620):
    html = m.get_root().render()
    encoded = base64.b64encode(html.encode("utf-8")).decode("ascii")
    display(HTML(
        f'<iframe src="data:text/html;base64,{encoded}" '
        f'width="100%" height="{height}" style="border:0;"></iframe>'
    ))


CENTER = gw.geocode("Washington, DC, USA")
CENTER

(38.8950982, -77.0363849)

## 1. Build the walking graph

`SpatialGraph` is the object we reuse for every query. The graph radius below is intentionally city-scale; reduce it while experimenting if you want faster first runs.

In [2]:
graph = gw.SpatialGraph.from_place(
    "Washington, DC, USA",
    network="walk",
    max_dist=13_000,
)

print(graph)
print(f"{graph.node_count():,} nodes, {graph.edge_count():,} edges")

SpatialGraph(nodes=259050, edges=788241, network_type=Walk)
259,050 nodes, 788,241 edges


## 2. Fetch grocery POIs

We use a large walking isochrone as a city-shaped fetch area, then filter OSM POIs to grocery-like tags. This keeps the example self-contained. For research use, you may want to bring your own grocery dataset and use Graphways only for the network reachability part.

In [3]:
coverage_iso = graph.isochrone(CENTER, minutes=[90])[0]
pois = graph.fetch_pois(coverage_iso)

GROCERY_SHOPS = {"supermarket", "grocery", "convenience", "greengrocer"}

groceries = [
    poi for poi in pois.pois
    if poi.tags.get("shop") in GROCERY_SHOPS
]

print(f"Fetched {pois.count:,} POIs")
print(f"Kept {len(groceries):,} grocery-like POIs")

for poi in groceries[:10]:
    print(poi.tags.get("name", "(unnamed)"), poi.tags.get("shop"), poi.lat, poi.lon)

Fetched 4,920 POIs
Kept 266 grocery-like POIs
Eagle Express convenience 38.9373316 -77.0890437
7-Eleven convenience 38.9453712 -77.0778046
7-Eleven convenience 38.8837515 -77.1029597
Manhattan Market convenience 38.9249001 -77.0519964
(unnamed) supermarket 38.9241523 -77.0522256
7-Eleven convenience 38.9053489 -77.0432258
Capital Harvest greengrocer 38.89456 -77.02969
New York Convenience convenience 38.8993189 -77.0328651
Harris Teeter supermarket 38.9218478 -77.0383963
Giant supermarket 38.9308461 -77.0318599


## 3. Snap grocery POIs to the graph

A grocery store is useful for walking access only if it snaps plausibly to the walking network. The snap distance guard filters out POIs whose coordinates are too far from the loaded graph.

In [4]:
grocery_targets = []

for poi in groceries:
    snap = graph.snap_point(poi.lat, poi.lon)
    if snap is None or snap.distance_m > 150:
        continue
    grocery_targets.append({
        "poi": poi,
        "node_id": snap.node_id,
        "snap_distance_m": snap.distance_m,
    })

print(f"{len(grocery_targets):,} groceries snapped within 150 m")

266 groceries snapped within 150 m


## 4. Sample analysis points

For a quick screen, this uses a regular grid over a rough DC bounding box. For a serious analysis, replace this with census block centroids or population-weighted points.

In [5]:
DC_BBOX = {
    "south": 38.79,
    "west": -77.12,
    "north": 38.995,
    "east": -76.91,
}

GRID_STEP_DEG = 0.01  # roughly 850-1100 m around DC

sample_points = []
lat = DC_BBOX["south"]
while lat <= DC_BBOX["north"]:
    lon = DC_BBOX["west"]
    while lon <= DC_BBOX["east"]:
        snap = graph.snap_point(lat, lon)
        if snap is not None and snap.distance_m <= 350:
            sample_points.append((lat, lon, snap.distance_m))
        lon += GRID_STEP_DEG
    lat += GRID_STEP_DEG

print(f"{len(sample_points):,} sampled points near the walking graph")

435 sampled points near the walking graph


## 5. Compute grocery access

For each sampled point, compute a 15-minute reachable graph. A point is marked as having grocery access if any snapped grocery node appears inside that reachable graph.

In [6]:
WALK_MINUTES = 15

results = []

for idx, (lat, lon, sample_snap_m) in enumerate(sample_points, start=1):
    reachable = graph.reachable((lat, lon), minutes=WALK_MINUTES, max_snap_m=350)

    grocery_times = [
        reachable.travel_time_to_node_id(target["node_id"])
        for target in grocery_targets
    ]
    grocery_times = [t for t in grocery_times if t is not None]

    results.append({
        "lat": lat,
        "lon": lon,
        "sample_snap_m": sample_snap_m,
        "has_access": bool(grocery_times),
        "nearest_grocery_min": min(grocery_times) / 60 if grocery_times else None,
        "reachable_groceries": len(grocery_times),
    })

access_rate = sum(r["has_access"] for r in results) / max(len(results), 1)
underserved = [r for r in results if not r["has_access"]]
served_minutes = [r["nearest_grocery_min"] for r in results if r["nearest_grocery_min"] is not None]

print(f"Sample points: {len(results):,}")
print(f"Access within {WALK_MINUTES} min: {access_rate:.1%}")
print(f"Likely underserved sample points: {len(underserved):,}")
if served_minutes:
    print(f"Median nearest grocery time among served points: {median(served_minutes):.1f} min")

Sample points: 435
Access within 15 min: 30.8%
Likely underserved sample points: 301
Median nearest grocery time among served points: 6.3 min


## 6. Map the result

Red cells are sample points with no grocery POI reachable within the walk-time threshold. Blue cells have at least one reachable grocery POI.

In [7]:
m = folium.Map(location=[CENTER[0], CENTER[1]], zoom_start=12, tiles="Cartodb Positron")

for target in grocery_targets:
    poi = target["poi"]
    folium.CircleMarker(
        location=[poi.lat, poi.lon],
        radius=4,
        color="#1b9e77",
        fill=True,
        fill_opacity=0.75,
        weight=1,
        tooltip=poi.tags.get("name", poi.tags.get("shop", "grocery")),
    ).add_to(m)

for row in results:
    color = "#2b6cb0" if row["has_access"] else "#d73027"
    tooltip = (
        f"reachable groceries: {row['reachable_groceries']}"
        if row["has_access"]
        else f"no grocery within {WALK_MINUTES} min"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.55,
        weight=0,
        tooltip=tooltip,
    ).add_to(m)

legend = branca.colormap.StepColormap(
    colors=["#d73027", "#2b6cb0"],
    index=[0, 1, 2],
    vmin=0,
    vmax=2,
    caption=f"Grocery access within {WALK_MINUTES} minutes",
)
legend.add_to(m)

show_map(m)

c:\Users\kylel\Rust\osm_graph\.venv\lib\site-packages\IPython\core\display.py:475: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Where to take this next

To turn this from a screening notebook into a real food-access study:

- replace the grid with census block or block-group centroids,
- weight each point by population,
- join ACS income and vehicle-access fields,
- use a curated grocery dataset if OSM coverage is incomplete,
- test several thresholds, such as 10, 15, and 20 minutes,
- compare walking, biking, transit, and driving assumptions separately.

Graphways handles the network-reachability layer; the policy definition and demographic joins should stay explicit in your analysis.